# Traffic Sign Visibility Audit

## Case question
Which traffic-sign images should be reviewed first, and which measurements support that priority? This notebook uses the labeled GTSRB training images to make a transparent, reproducible triage—not a classifier or an assessment of road safety. A flag means that an image merits human review, not that a sign is unsafe or mislabeled.

Run all cells from top to bottom. The raw files remain unchanged; the notebook writes derived tables and figures to `outputs/`. The project root or this `notebooks/` folder may be used as the working directory.

## 1. Setup and validation

Select five diverse sign classes, sample up to 500 images per class using seed 42, and validate the reusable pipeline. The supplied `src/audit.py` contains the actual processing functions and the `TrafficSignAudit` class; this notebook calls those functions rather than maintaining a second implementation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "audit.py").is_file() and (ROOT.parent / "src" / "audit.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "audit.py").is_file():
    raise FileNotFoundError("Start Jupyter from the project root or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
from audit import TrafficSignAudit, validate

DATA_DIR = ROOT / "data" / "GTSRB" / "Final_Training" / "Images"
OUTPUT_DIR = ROOT / "outputs"
CLASSES = [0, 1, 14, 17, 25]
SAMPLE_PER_CLASS = 500
SEED = 42
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Training images not found: {DATA_DIR}. See README.md.")
validate()
audit = TrafficSignAudit(DATA_DIR, OUTPUT_DIR, CLASSES, SAMPLE_PER_CLASS, SEED)

## 2. Inspect all annotation rows

The full-annotation class counts below use all available annotation rows, not the image sample. Data-quality diagnostics include data types, missing values, duplicate rows, invalid numeric fields, unavailable paths and class coverage. A warning is issued if an expected class is missing.

In [ ]:
audit.load()
print(f"Annotation rows: {audit.diagnostics['annotation_rows']:,}; "
      f"valid classes: {audit.diagnostics['unique_valid_classes']}")
display(pd.read_csv(audit.processed / "full_class_counts.csv"))
display(pd.Series(audit.diagnostics, name="value").to_frame())

## 3. Stratified sampling and image measurements

Each sampled image is read with OpenCV. Grayscale mean measures brightness (0–255), grayscale standard deviation measures contrast, Laplacian variance is a **proxy** for sharpness (squared response units), and minimum width/height measures image size in pixels. ROI coordinates are validated; the audit uses the full image for its rule. A small full-image/ROI comparison is saved. Failures are recorded in `skipped_images.csv` rather than silently dropped.

In [ ]:
audit.extract()
print(f"Sampled: {audit.diagnostics['sampled_rows']:,}; "
      f"processed: {audit.diagnostics['processed_images']:,}; "
      f"skipped: {audit.diagnostics['skipped_images']:,}")
print("ROI checks:", audit.diagnostics["roi_status_counts"])
print("Skip reasons:", audit.diagnostics["skip_reasons"])
display(audit.table.head())
display(pd.read_csv(audit.processed / "roi_comparison.csv"))

## 4. Review rule and sensitivity

For each feature the baseline threshold is its 10th percentile in the selected sample. A NumPy vectorized mask flags an image if **any** feature falls strictly below its own threshold. The 5th and 15th percentile alternatives test workload and ranking sensitivity. All thresholds are project-specific; they are not official safety standards.

In [ ]:
audit.summarize()
display(pd.DataFrame(audit.thresholds).T)
print("Overall share flagged by scenario:")
display(pd.Series(audit.diagnostics["flag_rates"], name="flag_rate").to_frame())
print("Sample measurements, flag rates and ranks by class:")
display(audit.summary)

## 5. Relationships and review examples

Spearman correlations summarize the association of brightness, contrast and sharpness. Correlation does not establish causation. The summary table includes sample sizes, measurements, three scenario flag rates and ranks, allowing comparisons alongside the full-annotation counts above. Class imbalance and repeated frames from the same sign limit generalization.

In [ ]:
correlations = pd.read_csv(audit.processed / "spearman_correlations.csv", index_col=0)
display(correlations)
audit.plot()  # Writes all six Matplotlib figures and the selected-example table.
display(pd.read_csv(audit.processed / "example_images.csv"))

### Contact sheet: flagged and unflagged images
The sheet shows six example images, their class labels and relevant measurements. Images are enlarged only for display. A small, legible image could be a false alarm, while glare or occlusion could escape this rule.

In [ ]:
display(Image(filename=str(audit.figures / "01_contact_sheet.png")))

### Brightness and sharpness distributions
The dashed lines indicate the 5th-, 10th- and 15th-percentile cutoffs for the selected sample.

In [ ]:
display(Image(filename=str(audit.figures / "02_distributions.png")))

### Sharpness by class
Boxplots compare the distributions; extreme values are omitted from the plot for readability, not removed from analysis.

In [ ]:
display(Image(filename=str(audit.figures / "03_sharpness_by_class.png")))

### Review rate by class
The baseline flag rate is the fraction of sampled images selected for human review, not the rate of unsafe signs.

In [ ]:
display(Image(filename=str(audit.figures / "04_flag_rate_by_class.png")))

### Contrast by class
The boxplots compare grayscale standard deviations in the selected classes.

In [ ]:
display(Image(filename=str(audit.figures / "05_contrast_by_class.png")))

### Contrast and sharpness
Each point represents a sampled image. The plot illustrates association, not a cause-and-effect relationship.

In [ ]:
display(Image(filename=str(audit.figures / "06_contrast_sharpness.png")))

## 6. Operational recommendation and limitations

Review flagged images first; within that queue, use the number of low-feature reasons as one transparent ordering aid. Review a random control sample of unflagged images to identify missed cases. Use human decisions, time per review and a representative local dataset to reevaluate thresholds. The five selected classes, sample cap, correlated frames from the same sign and proxy sharpness measure all limit generalization. The saved report provides a one-page decision memo and detailed interpretation.

**Data source:** German Traffic Sign Recognition Benchmark (GTSRB), Real-Time Computer Vision Group, Ruhr-Universität Bochum. [Benchmark information](https://benchmark.ini.rub.de/gtsrb_dataset.html); Stallkamp et al., IJCNN 2011. Data and derived outputs are stored separately.